# 1. Process PyNNLF Output: Ausgrid Solar Home Dataset (ASHD) vs Australian Centre for Advanced Photovoltaics (ACAP), Energy Data Platform (ACAP-EDP) 148 Households

## Purpose
Creates the shared 1-day comparison tables. It does not run PyNNLF.

## Inputs
- `RECAP_PATH`
- `CH5_DATA_PATH`

## Run Flow
1. Setup And Paths
2. Load And Validate ASHD 148hh Results
3. Load And Validate ACAP-EDP 148hh Results
4. Build And Export Comparison Tables
5. Debug Checklist

## Outputs
- `RESULTS_DIR / "ashd_acap_edp_148hh_fh8_combined_recap.csv"`
- `RESULTS_DIR / "ashd_acap_edp_148hh_fh8_nrmse_comparison.csv"`
- `RESULTS_DIR / "ashd_acap_edp_148hh_fh8_nrmse_stddev_comparison.csv"`
- `RESULTS_DIR / "paper_table_ashd_acap_edp_148hh_test_nrmse.csv"`
- `RESULTS_DIR / "paper_table_ashd_acap_edp_148hh_key_models_train_test_runtime.csv"`

## 1. Setup And Paths

Inputs are the publication recap for ASHD `ds20` and the Chapter 5 workbook rows for ACAP-EDP `ds11`.

In [ ]:
import os
from pathlib import Path
import sys

def find_publication_project(start: Path | None = None) -> Path:
    current = (start or Path.cwd()).resolve()
    for candidate in [current, *current.parents]:
        if (candidate / "specs").exists() and (candidate / "data").exists() and (candidate / "results").exists():
            return candidate
    raise FileNotFoundError("Could not find publication/journal_article_1 from the current working directory.")

def find_repo_root(start: Path) -> Path:
    for candidate in [start.resolve(), *start.resolve().parents]:
        if (candidate / "pyproject.toml").exists() and (candidate / "src" / "pynnlf").exists():
            return candidate
    raise FileNotFoundError("Could not find the PyNNLF repo root.")

PROJECT_DIR = find_publication_project()
REPO_ROOT = find_repo_root(PROJECT_DIR)
print(f"Publication project: {PROJECT_DIR}")
print(f"Repository root: {REPO_ROOT}")
import pandas as pd
RESULTS_DIR = PROJECT_DIR / "results" / "01_ashd_acap_edp_148hh_comparison"
RESULTS_DIR.mkdir(parents=True, exist_ok=True)
RECAP_PATH = PROJECT_DIR / "experiment_result" / "a1_experiment_result.csv"
def raw_data_root():
    """Locate the local source data tree, which is not distributed with this repository.

    Set the PYNNLF_RAW_DATA_DIR environment variable to the directory holding the
    source data folders before running this notebook.

    Returns:
        Path: root of the local source data tree.
    """
    root = os.environ.get("PYNNLF_RAW_DATA_DIR")
    if not root:
        raise RuntimeError(
            "PYNNLF_RAW_DATA_DIR is not set. Point it at your local source data "
            "directory; see the datasets documentation."
        )
    return Path(root)


RAW_DATA_ROOT = raw_data_root()

# Comparison workbook for the chapter tables; place it under the raw data root.
CH5_DATA_PATH = RAW_DATA_ROOT / "reference" / "ch5_data.xlsx"
MODEL_ORDER = ['m1_naive_hp1', 'm2_snaive_hp2', 'm3_ets_hp1', 'm4_arima_hp1', 'm6_lr_hp1', 'm7_ann_hp1', 'm8_dnn_hp1', 'm9_rt_hp3', 'm10_rf_hp1', 'm13_lstm_hp2', 'm16_prophet_hp1', 'm17_xgb_hp1']
FH8_MINUTES = 1440
print(f"Results directory: {RESULTS_DIR}")

## 2. Load And Validate ASHD 148hh Results

In [ ]:
if not RECAP_PATH.exists():
    raise FileNotFoundError(f"Missing recap at {RECAP_PATH}. Run the ASHD experiment notebook first.")
recap = pd.read_csv(RECAP_PATH)
required = {"dataset_no", "forecast_horizon_min", "model_name", "train_nRMSE", "test_nRMSE", "test_nRMSE_stddev", "runtime_ms"}
missing = required - set(recap.columns)
if missing:
    raise ValueError(f"Recap is missing required columns: {sorted(missing)}")
ashd = recap.loc[recap["dataset_no"].eq("ds20") & pd.to_numeric(recap["forecast_horizon_min"], errors="coerce").eq(FH8_MINUTES)].copy()
ashd["forecast_horizon_min"] = FH8_MINUTES
ashd["model_name"] = pd.Categorical(ashd["model_name"], categories=MODEL_ORDER, ordered=True)
ashd["dataset_label"] = "ASHD_148hh_weather"
ashd["result_source"] = "publication_journal_article_1_experiment_result"
ashd = ashd.sort_values("model_name")
missing_models = sorted(set(MODEL_ORDER) - set(ashd["model_name"].astype(str)))
if missing_models:
    raise ValueError(f"ASHD ds20 1-day result is missing models: {missing_models}")
if ashd.shape[0] != len(MODEL_ORDER):
    raise ValueError(f"Expected {len(MODEL_ORDER)} ASHD rows, found {ashd.shape[0]}")
display(ashd[["dataset_no", "forecast_horizon_min", "model_name", "test_nRMSE", "test_nRMSE_stddev"]])

## 3. Load And Validate ACAP-EDP 148hh Results

ACAP-EDP is imported from the current Chapter 5 workbook and is not rerun here.

In [ ]:
if not CH5_DATA_PATH.exists():
    raise FileNotFoundError(CH5_DATA_PATH)
acap_edp_sheet = pd.read_excel(CH5_DATA_PATH, sheet_name="3.3.1 dataset")
horizon = pd.to_numeric(acap_edp_sheet["forecast_horizon_min"], errors="coerce")
dataset_no = acap_edp_sheet["dataset_no"].astype(str).str.strip()
acap_edp = acap_edp_sheet.loc[dataset_no.eq("ds11") & horizon.eq(FH8_MINUTES)].copy()
acap_edp["forecast_horizon_min"] = FH8_MINUTES
if acap_edp.empty:
    raise ValueError("Could not find ACAP-EDP ds11 1-day rows in ch5_data.xlsx sheet '3.3.1 dataset'.")
acap_edp["model_name"] = pd.Categorical(acap_edp["model_name"], categories=MODEL_ORDER, ordered=True)
acap_edp["dataset_label"] = "ACAP-EDP_148hh_weather"
acap_edp["result_source"] = "ch5_data_xlsx_3.3.1_dataset"
acap_edp = acap_edp.sort_values("model_name")
missing_models = sorted(set(MODEL_ORDER) - set(acap_edp["model_name"].astype(str)))
if missing_models:
    raise ValueError(f"ACAP-EDP ds11 1-day result is missing models: {missing_models}")
if acap_edp.shape[0] != len(MODEL_ORDER):
    raise ValueError(f"Expected {len(MODEL_ORDER)} ACAP-EDP rows, found {acap_edp.shape[0]}")
display(acap_edp[["dataset_no", "forecast_horizon_min", "model_name", "test_nRMSE", "test_nRMSE_stddev"]])

## 4. Build And Export Comparison Tables

In [ ]:
def wide_metric_by_dataset(df, metric_column):
    table = df.pivot_table(index="model_name", columns="dataset_label", values=metric_column, aggfunc="first", observed=False)
    table = table.reindex(index=MODEL_ORDER)
    table = table[["ASHD_148hh_weather", "ACAP-EDP_148hh_weather"]]
    table.index.name = "model_hp"
    table.columns.name = None
    return table

PAPER_MODEL_ORDER = ['m17_xgb_hp1', 'm8_dnn_hp1', 'm6_lr_hp1', 'm10_rf_hp1', 'm7_ann_hp1', 'm1_naive_hp1', 'm3_ets_hp1', 'm9_rt_hp3', 'm13_lstm_hp2', 'm2_snaive_hp2', 'm16_prophet_hp1', 'm4_arima_hp1']
PAPER_KEY_MODELS = ['m17_xgb_hp1', 'm1_naive_hp1', 'm4_arima_hp1']
PAPER_MODEL_LABELS = {'m1_naive_hp1': 'naive_hp1', 'm2_snaive_hp2': 'snaive_hp2', 'm3_ets_hp1': 'ets_hp1', 'm4_arima_hp1': 'arima_hp1', 'm6_lr_hp1': 'lr_hp1', 'm7_ann_hp1': 'ann_hp1', 'm8_dnn_hp1': 'dnn_hp1', 'm9_rt_hp3': 'rt_hp3', 'm10_rf_hp1': 'rf_hp1', 'm13_lstm_hp2': 'lstm_hp2', 'm16_prophet_hp1': 'prophet_hp1', 'm17_xgb_hp1': 'xgb_hp1'}

def short_model_name(model_name):
    return PAPER_MODEL_LABELS.get(str(model_name), str(model_name))

common_columns = ["experiment_no", "exp_date", "dataset_no", "dataset_label", "forecast_horizon_min", "model_no", "hyperparameter_no", "model_name", "runtime_ms", "train_RMSE", "train_RMSE_stddev", "test_RMSE", "test_RMSE_stddev", "train_nRMSE", "train_nRMSE_stddev", "test_nRMSE", "test_nRMSE_stddev", "result_source"]
for frame in [ashd, acap_edp]:
    for column in common_columns:
        if column not in frame.columns:
            frame[column] = pd.NA
combined = pd.concat([ashd[common_columns], acap_edp[common_columns]], ignore_index=True)
combined["model_name"] = pd.Categorical(combined["model_name"], categories=MODEL_ORDER, ordered=True)
combined = combined.sort_values(["dataset_label", "model_name"])
comparison_nrmse = wide_metric_by_dataset(combined, "test_nRMSE")
comparison_stddev = wide_metric_by_dataset(combined, "test_nRMSE_stddev")

def get_single_value(frame, dataset_label, model_name, column):
    match = frame.loc[frame["dataset_label"].eq(dataset_label) & frame["model_name"].astype(str).eq(model_name), column]
    if match.empty:
        raise ValueError(f"Missing {column} for {dataset_label} / {model_name}")
    return pd.to_numeric(match.iloc[0], errors="coerce")

paper_table2 = (
    comparison_nrmse
    .reindex(PAPER_MODEL_ORDER)
    .rename(index=short_model_name, columns={"ASHD_148hh_weather": "ASHD", "ACAP-EDP_148hh_weather": "ACAP-EDP"})
    .sort_values("ASHD", ascending=True, kind="mergesort")
    .round(1)
)

paper_rows = []
for model in PAPER_KEY_MODELS:
    row = {"Model Name": short_model_name(model)}
    for dataset_label, dataset_name in [("ACAP-EDP_148hh_weather", "ACAP-EDP"), ("ASHD_148hh_weather", "ASHD")]:
        row[f"{dataset_name} Train nRMSE (%)"] = get_single_value(combined, dataset_label, model, "train_nRMSE")
        row[f"{dataset_name} Test nRMSE (%)"] = get_single_value(combined, dataset_label, model, "test_nRMSE")
        row[f"{dataset_name} Training Time (s)"] = get_single_value(combined, dataset_label, model, "runtime_ms") / 1000.0
    paper_rows.append(row)
paper_table3 = pd.DataFrame(paper_rows)
for column in paper_table3.columns.drop("Model Name"):
    paper_table3[column] = pd.to_numeric(paper_table3[column], errors="coerce").round(1)

combined.to_csv(RESULTS_DIR / "ashd_acap_edp_148hh_fh8_combined_recap.csv", index=False)
comparison_nrmse.to_csv(RESULTS_DIR / "ashd_acap_edp_148hh_fh8_nrmse_comparison.csv")
comparison_stddev.to_csv(RESULTS_DIR / "ashd_acap_edp_148hh_fh8_nrmse_stddev_comparison.csv")
paper_table2.to_csv(RESULTS_DIR / "paper_table_ashd_acap_edp_148hh_test_nrmse.csv")
paper_table3.to_csv(RESULTS_DIR / "paper_table_ashd_acap_edp_148hh_key_models_train_test_runtime.csv", index=False)

display(comparison_nrmse.round(3))
display(comparison_stddev.round(3))
display(paper_table2)
display(paper_table3)

## 5. Debug Checklist

If this fails, check the ASHD recap exists and the workbook still has `ds11`, 1440-minute rows.